<a href="https://colab.research.google.com/github/meryambutt123-a11y/code-switching-codesaviours-si26-maryam/blob/main/SI26_Week7_maryam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install required libraries
!pip install -q transformers torch datasets seqeval

import pandas as pd
import torch
from google.colab import drive
from sklearn.model_selection import train_test_split

# 2. Mount Google Drive
drive.mount('/content/drive')

# 3. Load dataset from your exact folder path
# (Assuming you put dataset.csv in your Project2_Data folder!)
df = pd.read_csv('/content/drive/MyDrive/Project2_Data/dataset.csv')

# 4. Create label mapping for the model
label2id = {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label = {0: 'URD', 1: 'ENG', 2: 'MIX'}

# 5. Group the flat rows back into complete sentences
sentences = df.groupby('sentence').apply(
    lambda x: {'words': x['word'].tolist(), 'labels': x['label'].tolist()}
).tolist()

# 6. Split into train (80%) and test (20%) sets
train_data, test_data = train_test_split(sentences, test_size=0.2, random_state=42)

# Verify sentence counts
print(f'Training sentences: {len(train_data)}')
print(f'Testing sentences: {len(test_data)}')

Mounted at /content/drive
Training sentences: 120
Testing sentences: 30


/tmp/ipykernel_803/3998831896.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby('sentence').apply(


In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import Dataset

# 1. Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

# 2. Convert our Python lists into HuggingFace Datasets
train_hf = Dataset.from_list(train_data)
test_hf = Dataset.from_list(test_data)

# 3. The Alignment Function
def tokenize_and_align_labels(examples):
    # Tokenize the words
    tokenized_inputs = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128
    )

    labels = []
    # Loop through each sentence's labels
    for i, label_list in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens back to words
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens (like [CLS] or [SEP]) get -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # The first token of a real word gets its actual mapped ID (0, 1, or 2)
                label_ids.append(label2id[label_list[word_idx]])
            else:
                # Any extra subtokens from the same word get -100
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Tokenize and align the data!
# Using batched=True processes multiple elements at once to speed things up
tokenized_train = train_hf.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_hf.map(tokenize_and_align_labels, batched=True)

# 5. Load the XLM-RoBERTa Model
# We set num_labels=3 because we have URD, ENG, and MIX
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("\nModel and Tokenizer loaded successfully, and tokens are perfectly aligned!")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Model and Tokenizer loaded successfully, and tokens are perfectly aligned!


In [5]:
import numpy as np
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer

# 1. Initialize the Data Collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 2. Define the Evaluation Metrics Function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove the -100 index (ignored special tokens) to get accurate scores
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

# 3. Configure Training Arguments (5 epochs, batch size of 16)
training_args = TrainingArguments(
    output_dir="./xlm-roberta-code-switching",
    eval_strategy="epoch",  # Evaluate at the end of every epoch
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    push_to_hub=False,
)

# 4. Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer, # <--- THIS IS THE FIX!
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 5. Execute Training!
print("Starting the training loop...")
trainer.train()

# 6. Print Final Evaluation Scores
print("\nFinal Model Evaluation:")
trainer.evaluate()

Starting the training loop...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.836478,0.000000,0.000000,0.000000,0.604061
2,No log,0.722230,0.000000,0.000000,0.000000,0.604061
3,No log,0.666200,0.348837,0.125000,0.184049,0.649746
4,No log,0.619140,0.727273,0.533333,0.615385,0.796954
5,No log,0.585501,0.747368,0.591667,0.660465,0.817259


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: MIX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: URD seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ENG seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))



Final Model Evaluation:


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
No log,0.585501,5,0.747368,0.591667,0.660465,0.817259


{'eval_loss': 0.5855007767677307,
 'eval_precision': 0.7473684210526316,
 'eval_recall': 0.5916666666666667,
 'eval_f1': 0.6604651162790698,
 'eval_accuracy': 0.817258883248731}

In [7]:
from huggingface_hub import notebook_login

# Enter your Hugging Face write token when prompted
notebook_login()

In [8]:
# Push model and tokenizer to your Hugging Face account
repo_name = "xlm-roberta-code-switching-si26"

trainer.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print(f"Model and tokenizer successfully pushed to {repo_name}!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp4ot9t4ba/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model and tokenizer successfully pushed to xlm-roberta-code-switching-si26!
